# Cayley-Dickson Construction
This project is to represent the Cayley-Dickson construction for arbitrary base fields. <br>
When the base field is $\mathbb{R}$, this leads to the complex numbers, quaternions, octonions, etc.


In [9]:
import numpy as np

In [46]:
class Cayley:
    ''' A class to encode an array of numbers into a Cayley number.
        A CD_Alg object has the following attributes:
        coeff -  the array of coefficients as a numpy array. 
                
        field - the field of the coefficients
                'R' = real numbers, 0 = integers, p (prime)= finite field. 
                
        degree - the number of successive quadratic extensions needed to 
                represent the number. This is the smallest integer such 
                that 2^degree >= len(coeff). For example, the number 1
                has degree 0, the number 1 + i has degree 1, the number
                1 + i + j has degree 2, and so on. 
                
        real - the real part of the Cayley number.   '''
    
    def __init__(self, coeff=np.array([1]), degree=None, field='R'):
        if len(coeff) == 0: # edge case for empty array
            degree = 0
        if degree == None: # if degree not specified, use the smallest possible
            degree = (len(coeff) - 1).bit_length() # smallest extension needed
        if len(coeff) > (1 << degree): # check that it is large enough
            raise ValueError('Degree too small for number of coefficients')
        # pad with zeros
        coeff = np.pad(coeff, (0, (1 << degree) - len(coeff)), 'constant')

        self.coeff = coeff
        self.field = field
        self.degree = degree
        self.real = coeff[0]

    def __repr__(self):
        i = chr(0x0001D48A) # unicode for bold italic i
        j = chr(0x0001D48B) # unicode for bold italic j
        k = chr(0x0001D48C) # unicode for bold italic k
        def display(n):
            if n == 0:
                return None
            elif n == 1:
                return ' + '
            elif n == -1:
                return ' - '
            elif n < 0:
                return f' - {-1 * n} '
            else:
                return f' + {n} '
        if self.degree == 0:
            return f'{self.coeff[0]}'
        elif self.degree == 1:
            if self.coeff[1] != 0:
                if self.coeff[0] != 0:
                    return f'{self.coeff[0]}{display(self.coeff[1])}{i}'
                return f'{self.coeff[1]}{i}'
            return '0'
    
        elif self.degree == 2:
            if self.coeff[0] != 0:
                string = f'{self.coeff[0]}'
            else:
                string = ''
            if self.coeff[1] != 0:
                string += f'{display(self.coeff[1])}{i}'
            if self.coeff[2] != 0:
                string += f'{display(self.coeff[2])}{j}'
            if self.coeff[3] != 0:
                string += f'{display(self.coeff[3])}{k}'
            return string
        else:
            def subscript(n):                
                # Base character 'i' in subscript Unicode
                base_char = chr(0x1D48A)
                
                # Unicode subscripts for digits 0-9
                subscripts = {
                    '0': chr(0x2080),
                    '1': chr(0x2081),
                    '2': chr(0x2082),
                    '3': chr(0x2083),
                    '4': chr(0x2084),
                    '5': chr(0x2085),
                    '6': chr(0x2086),
                    '7': chr(0x2087),
                    '8': chr(0x2088),
                    '9': chr(0x2089),
                }
                
                # Convert n into its subscript form
                subscript_digits = ''.join(subscripts[digit] for digit in str(n))
                
                # Return the result
                return base_char + subscript_digits
            string = f'{self.coeff[0]}'
            for digit in range(1, 1 << self.degree):
                if self.coeff[digit] != 0:
                    string += f'{display(self.coeff[digit])}{subscript(digit)}'
            return string
    def left(self): # left half of coefficients as a Cayley number
        if self.degree == 0: # base case
            return self
        left_coeff = self.coeff[:1 << (self.degree - 1)]
        left_degree = self.degree - 1
        return Cayley(left_coeff, left_degree, self.field)
    
    def right(self): # right half of coefficients as a Cayley number
        if self.degree == 0: # base case
            return Cayley([0], 0, self.field)
        right_coeff = self.coeff[1 << (self.degree - 1):]
        right_degree = self.degree - 1
        return Cayley(right_coeff, right_degree, self.field)

    def conj(self): # conjugate of a Cayley number (a, b)* = (a*, -b)
        if self.degree == 0: # base case
            return self
        else:
            if self.field in ['R', 'Z', 0]: # characteristic 0 case
                conj_coeff = self.coeff.copy() * -1 
                conj_coeff[0] *= -1
            else: # characteristic p case
                p = self.field
                conj_coeff = (self.coeff.copy() * -1) % p
                conj_coeff[0] = (conj_coeff[0] * -1) % p
            return Cayley(conj_coeff, self.degree, self.field)

    def __eq__(self, other):
        eq_coeffs = (self.coeff == other.coeff)
        eq_field = (self.field == other.field)
        return eq_coeffs and eq_field
    
    def __add__(self, other):
        if self.field != other.field:
            raise ValueError('Fields must match')
        sum_coeff = np.add(self.coeff, other.coeff)
        return Cayley(sum_coeff, self.degree, self.field)
    
    def __sub__(self, other):
        if self.field != other.field:
            raise ValueError('Fields must match')
        diff_coeff = np.subtract(self.coeff, other.coeff)
        return Cayley(diff_coeff, self.degree, self.field)
    
    def __mul__(self, other):
        if type(other) in [float, int]: # Scalar multiplication
            if type(other) == int and self.field not in ['R', 0]:
                p = self.field  # characteristic p case
                return Cayley((self.coeff * other) % p, self.field)
            return Cayley(self.coeff * other, self.field) # characteristic 0
        
        elif type(other) == Cayley: # Cayley number multiplication
            if self.field != other.field: # base fields should match
                raise ValueError('Fields must match to multiply')
            if self.degree == 0:    # base case is when the degree is 0
                return Cayley(other.coeff * self.coeff[0],
                                other.degree, self.field)
            elif other.degree == 0: 
                return Cayley(self.coeff * other.coeff[0],
                               self.degree, self.field)
            else: # break into smaller Cayley numbers for recursive multiplication
                D = max(self.degree, other.degree) # make sure the degrees match
                # multiplication formula is given by
                # (a_L, a_R)(b_L, b_R) = (a_L b_L - b_R* a_R, b_R a_L + a_R b_L* )
                a_L = Cayley(self.left().coeff, D - 1, self.field)
                a_R = Cayley(self.right().coeff, D - 1, self.field)
                b_L = Cayley(other.left().coeff, D - 1, other.field)
                b_R = Cayley(other.right().coeff, D - 1, other.field)

                prod_right = b_R * a_L + a_R * b_L.conj()  # recursive call 
                prod_left = a_L * b_L - b_R.conj() * a_R        
                product_coeff = np.concatenate((prod_left.coeff, prod_right.coeff))
                return Cayley(product_coeff, D, self.field)
        else:
            raise ValueError('Multiplication not defined for these types')
        
q = Cayley([1, 1, 1, 1], 2, 'R')
p = Cayley([1, 1, 1, 1])

r = q * p
r.coeff

array([-2,  2,  2,  2])

In [53]:
a =Cayley([0.1,1,0,-1,0,-3,-.1,0,0,1], degree=None, field='R')
a

0.1 + 𝒊₁ - 𝒊₃ - 3.0 𝒊₅ - 0.1 𝒊₆ + 𝒊₉

In [12]:
chr(0x0001D48A) + chr(0x2084)

'𝒊₄'

In [42]:
A = np.array([1,2,3])
A = np.pad(A, (0, 4 - len(A)), 'constant')
A

array([1, 2, 3, 0])